In [2]:
import torch
import torch_geometric.data.data

# Register the specific class mentioned in your error
torch.serialization.add_safe_globals([torch_geometric.data.data.DataEdgeAttr])

# You might also need the standard Data class
torch.serialization.add_safe_globals([torch_geometric.data.data.Data])


In [4]:
import torch
from ogb.nodeproppred import PygNodePropPredDataset

# 1. Save the original torch.load function
_original_load = torch.load

# 2. Define a wrapper that forces weights_only=False
def unsafe_load(*args, **kwargs):
    # Force weights_only to False to allow loading legacy OGB files
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)

# 3. Patch torch.load temporarily
torch.load = unsafe_load

print("Loading OGB dataset with legacy support...")
try:
    # Now this call will use the unsafe_load internally
    dataset_node = PygNodePropPredDataset(name='ogbn-arxiv', root='./data')
    print("Success!")
finally:
    # 4. Restore the original torch.load (Important!)
    torch.load = _original_load

# Verify data is loaded
data = dataset_node[0]
print(data)

Loading OGB dataset with legacy support...
Success!
Data(num_nodes=169343, edge_index=[2, 1166243], x=[169343, 128], node_year=[169343, 1], y=[169343, 1])


In [1]:
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn

# Configuration parameters
n_qubits = 4
n_layers = 2
noise_prob = 0.01  # Depolarizing noise probability
n_shots = 1000  # Number of shots for measurement

# Create device with depolarizing noise on default.mixed
dev = qml.device('default.mixed', wires=n_qubits, shots=n_shots)

# Define a simple Parameterized Quantum Circuit (PQC)
@qml.qnode(dev, interface='torch', diff_method='parameter-shift')
def qnn_circuit(inputs, weights):
    """
    Simple PQC with:
    - Angle encoding for inputs
    - Trainable rotation layers
    - Entangling CNOT gates
    - Depolarizing noise after each operation
    """
    # Encode inputs with depolarizing noise
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
        qml.DepolarizingChannel(noise_prob, wires=i)
    
    # Parameterized layers
    for layer in range(n_layers):
        # Rotation gates
        for i in range(n_qubits):
            qml.RX(weights[layer, i, 0], wires=i)
            qml.DepolarizingChannel(noise_prob, wires=i)
            qml.RY(weights[layer, i, 1], wires=i)
            qml.DepolarizingChannel(noise_prob, wires=i)
            qml.RZ(weights[layer, i, 2], wires=i)
            qml.DepolarizingChannel(noise_prob, wires=i)
        
        # Entangling layer
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
            qml.DepolarizingChannel(noise_prob, wires=i)
            qml.DepolarizingChannel(noise_prob, wires=i + 1)
        
        # Connect last to first qubit
        qml.CNOT(wires=[n_qubits - 1, 0])
        qml.DepolarizingChannel(noise_prob, wires=n_qubits - 1)
        qml.DepolarizingChannel(noise_prob, wires=0)
    
    # Measure expectations of Pauli-Z on all qubits
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Get weight shapes for the QNN
weight_shapes = {"weights": (n_layers, n_qubits, 3)}

print(f"QNN Configuration:")
print(f"  Device: default.mixed")
print(f"  Qubits: {n_qubits}")
print(f"  Layers: {n_layers}")
print(f"  Depolarizing noise probability: {noise_prob}")
print(f"  Shots: {n_shots}")
print(f"  Weight shape: {weight_shapes['weights']}")


QNN Configuration:
  Device: default.mixed
  Qubits: 4
  Layers: 2
  Depolarizing noise probability: 0.01
  Shots: 1000
  Weight shape: (2, 4, 3)


In [3]:
# Create TorchLayer wrapper for the QNN
class QuantumNeuralNetwork(nn.Module):
    def __init__(self, n_qubits, n_layers, noise_prob=0.01, n_shots=1000):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.noise_prob = noise_prob
        self.n_shots = n_shots
        
        # Create device (can be recreated with different shots)
        self._create_device()
        
        # Create the quantum layer using TorchLayer
        self.qlayer = qml.qnn.TorchLayer(qnn_circuit, weight_shapes)
        
    def _create_device(self):
        """Create/recreate device with current shot settings"""
        global dev
        dev = qml.device('default.mixed', wires=self.n_qubits, shots=self.n_shots)
    
    def set_shots(self, n_shots):
        """Change the number of shots and recreate the device"""
        self.n_shots = n_shots
        self._create_device()
        # Recreate the quantum layer with new device
        self.qlayer = qml.qnn.TorchLayer(qnn_circuit, weight_shapes)
        print(f"Updated shots to {n_shots}")
    
    # def set_noise_prob(self, noise_prob):
    #     """Change the depolarizing noise probability"""
    #     global noise_prob as global_noise_prob
    #     self.noise_prob = noise_prob
    #     # Note: Need to recreate circuit for this to take effect
    #     print(f"Updated noise probability to {noise_prob}")
    
    def forward(self, x):
        # Ensure input has the right number of features
        if x.shape[-1] != self.n_qubits:
            # Apply a linear transformation if input dimension doesn't match
            if not hasattr(self, 'input_projection'):
                self.input_projection = nn.Linear(x.shape[-1], self.n_qubits).to(x.device)
            x = self.input_projection(x)
        
        # Apply quantum circuit
        return self.qlayer(x)

# Instantiate the QNN
qnn = QuantumNeuralNetwork(n_qubits, n_layers, noise_prob, n_shots)

print(f"\nQuantumNeuralNetwork created!")
print(f"  Parameters: {sum(p.numel() for p in qnn.parameters())} trainable weights")
print(f"  Input size: {n_qubits}")
print(f"  Output size: {n_qubits}")



QuantumNeuralNetwork created!
  Parameters: 24 trainable weights
  Input size: 4
  Output size: 4


In [4]:
# Test the QNN with sample data
print("="*60)
print("Testing the Quantum Neural Network")
print("="*60)

# Create sample input
batch_size = 3
sample_input = torch.randn(batch_size, n_qubits)

print(f"\nInput shape: {sample_input.shape}")
print(f"Input data:\n{sample_input}")

# Forward pass
output = qnn(sample_input)

print(f"\nOutput shape: {output.shape}")
print(f"Output data:\n{output}")

print("\n" + "="*60)
print("Testing with different shot numbers")
print("="*60)

# Test with different shot configurations
for shots in [100, 500, 1000, 5000]:
    qnn.set_shots(shots)
    output = qnn(sample_input)
    print(f"Shots={shots:5d} | Output mean: {output.mean().item():.4f} | Output std: {output.std().item():.4f}")


Testing the Quantum Neural Network

Input shape: torch.Size([3, 4])
Input data:
tensor([[-0.1338, -0.6765,  0.9113, -1.1804],
        [ 0.7175, -0.6042,  0.6753, -1.6373],
        [-0.3741,  0.4572,  1.3080, -1.4868]])


IndexError: index 3 is out of bounds for dimension 0 with size 3

In [ ]:
# Visualize the quantum circuit
print("="*60)
print("Quantum Circuit Visualization")
print("="*60)

# Draw the circuit with sample inputs
sample_weights = torch.randn(n_layers, n_qubits, 3)
sample_single_input = torch.randn(n_qubits)

fig, ax = qml.draw_mpl(qnn_circuit)(sample_single_input, sample_weights)
print("\nCircuit diagram generated (displaying architecture with depolarizing noise)")


In [ ]:
# Example: Training the QNN on a simple task
print("="*60)
print("Training Example: Simple Classification Task")
print("="*60)

# Create a simple binary classification dataset
n_samples = 20
X_train = torch.randn(n_samples, n_qubits)
y_train = torch.randint(0, 2, (n_samples,)).float()

# Define loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(qnn.parameters(), lr=0.01)

# Create a simple classifier with QNN
class QNNClassifier(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = qnn
        self.fc = nn.Linear(n_qubits, 1)
    
    def forward(self, x):
        x = self.qnn(x)
        x = self.fc(x)
        return x.squeeze()

model = QNNClassifier(qnn)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

# Training loop
print("\nTraining for 5 epochs...")
for epoch in range(5):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    # Calculate accuracy
    predictions = (torch.sigmoid(outputs) > 0.5).float()
    accuracy = (predictions == y_train).float().mean()
    
    print(f"Epoch {epoch+1}/5 | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.4f}")

print("\nTraining complete!")
